# Gemma3-4B -- SFT on ViNumQA + FinQA, **program-only labels** (train only)

Fine-tune `unsloth/gemma-3-4b-it` to emit the gold FinQA-style computation program directly: no
`<think>` block, no reasoning trace. This is the Gemma counterpart of
`qwen3-4b-stf-wo-reasoning-trace.ipynb` in this folder.

**This notebook only trains and saves the LoRA adapter.** Scoring lives in
`gemma3-4b-stf-wo-reasoning-eval-only.ipynb`; add this notebook's output as a data source there.
The split exists because training and generation each want the whole GPU: holding optimiser state,
gradients and a 4-bit base model in one session and then running 497 generations in the same kernel
is how a T4 runs out of memory at the last possible moment, after the expensive part is already done.

## Training data: ViNumQA train + FinQA, de-duplicated

Training pools two corpora, which share a record schema (`id` / `pre_text` / `table` / `post_text` /
`qa{question, program, exe_ans}`) and so need no per-source handling:

| source | rows |
|---|---|
| `ViNumQA/train.json` (Vietnamese) | 2 993 |
| `FinQA_processed.json` (English) | 8 281 |
| pooled | 11 274 |
| **minus rows whose `id` is in valid/test** (all 486 from FinQA) | **10 788** |
| minus rows over `MAX_SEQ_LENGTH = 2048` (1 072, 9.9 %) | **9 716** |

`valid.json` (584, of which 490 survive the length filter) and `test.json` (497) remain
ViNumQA-only, so evaluation stays Vietnamese.

### Train/test contamination -- now removed, not just reported

ViNumQA is a Vietnamese translation of FinQA and the two **share `id`s**, so FinQA contained an
English twin -- same table, same question, same gold program -- of a large slice of the eval splits:
210 of the 497 test rows (42.3 %) and 276 of the 584 valid rows (47.3 %).

Earlier revisions printed those counts and trained on the rows anyway, which made the resulting test
PA/EA partly a memorisation measurement and not comparable to the other Table 2 rows. **The data cell
now drops them** (11 274 -> 10 788, verified against the real files) and asserts the overlap is empty
afterwards. It also means `eval_loss` is no longer inflated by rows the model has memorised -- which
matters more here than before, because `eval_loss` now drives both checkpoint selection and early
stopping.

## Schedule

Hyperparameters follow `qwen3-4b-stf-wo-reasoning-trace.ipynb`: effective batch 16 (2 x 8), lr 2e-4,
cosine, warmup 0.03, `adamw_8bit`, weight decay 0.001, seed 3407, `save_total_limit=2`,
`logging_steps=20`, LoRA `r=32 / alpha=32 / dropout=0`, `MAX_SEQ_LENGTH=2048`, same
`SYSTEM_MESSAGE` / `USER_MESSAGE_FRAME`, same `train_on_responses_only`.

The eval / checkpoint / stopping schedule is **step-based**, matching
`sft-w-reasoning-trace-distill/gemma3-4b-stf-w-reasoning-trace.ipynb` rather than the Qwen baseline:

| | Qwen baseline | here |
|---|---|---|
| `num_train_epochs` | 3 | **1** |
| eval | `"epoch"` (3 passes) | **`"steps"`, `eval_steps=20`** (~30 passes) |
| save | `"epoch"` | **`"steps"`, `save_steps=20`** |
| early stopping | none | **`patience=3`, `threshold=0.0`** |
| `load_best_model_at_end` | `True` on `eval_loss` | `True` on `eval_loss` |

**Two consequences to keep in view, both surfaced by the notebook itself rather than left implicit:**

1. **Eval is the largest cost after training.** ~30 passes over the 490-row valid set is ~7 350 extra
   forward batches, roughly +40 % on top of training time. The trainer cell prints the projection
   before training starts. `EVAL_SUBSET_SIZE` in the dataset cell is the cheapest lever if that is
   too much -- 128 cuts eval cost ~74 % and keeps `eval_loss` a consistent signal, because the
   subset is fixed by seed and identical at every eval.
2. **`patience=3` at `eval_steps=20` stops after 60 steps without improvement** -- about 10 % of this
   epoch. The same settings on the sibling notebook's `pa_en_finqa` variant stopped at step 220 of
   456 with the best checkpoint at 160, i.e. the run trained on ~35 % of the data. The stats cell
   after training prints `global_step` against the planned count so this is visible, and it belongs
   in the write-up: a run cut short is not comparable to the Qwen baseline, which has no early
   stopping and always consumes its whole file. Raising `eval_steps` to 150 makes `patience=3` mean
   450 steps instead of 60, if that turns out to be the problem.

Two further settings are pure throughput and change nothing about which samples are seen:
`group_by_length=True` (buckets similar lengths so batch=2 stops padding to a random partner) and
`dataset_num_proc=2`. Set `GROUP_BY_LENGTH = False` in the trainer cell to drop the first, at the
cost of roughly 15-20 % runtime.

### Rows longer than `MAX_SEQ_LENGTH` are dropped, not truncated

TRL truncates from the **right**, and the assistant turn -- the gold program, the only span this run
computes loss on -- sits at the very end. A truncated row therefore arrives with every label masked
to `-100`; a batch where both rows are like that makes the mean cross-entropy `NaN` and the LoRA
weights never recover. So the dataset cell removes those rows and prints the counts, and the masking
cell scans 500 random examples for any that slipped through.

**The cap is worth a second look before you run.** Measured on the real files with this tokenizer:

| cap | train over | valid over |
|---|---|---|
| **2048** (current, = baseline) | 1 072 (9.9 %) | 94 (16.1 %) |
| 2560 | 190 (1.8 %) | 15 (2.6 %) |
| 3072 | 58 (0.5 %) | 2 (0.3 %) |
| 5678 (previous) | 0 | 0 -- the longest row is 4 186 tokens |

Because TRL pads to the longest row **in the batch** rather than to the cap, raising it barely costs
anything: mean length moves from ~1 500 (at 2048, after the drops) to 1 613 uncapped. **2560 buys
back 882 training rows and 79 valid rows for roughly 5 % more runtime.** 2048 is kept here because
it is the baseline's value -- but it is one number in the model-loading cell if you would rather
keep the data.

## Gemma-specific mechanics

None of these are stylistic. Each was an outright failure -- a crash, a hang or a silent no-op --
when the Qwen setting was carried over to `gemma-3-4b-it`:

1. **`UNSLOTH_ENABLE_FLEX_ATTENTION=0` + `attn_implementation="eager"`,** set *before* `import
   unsloth`. gemma3 is in Unsloth's `_FLEX_PREFERRED_MODELS`; that kernel does not compile on a T4
   (sm75) and the `sdpa_dense` fallback materialises the full `(B, H, L, L)` score matrix. SDPA is
   not reachable either (gemma3 is in `DISABLE_SDPA_MODEL_NAMES`), so eager is not a downgrade.
2. **`FastModel` + `finetune_*` flags, not `FastLanguageModel` + `target_modules`.**
   `unsloth/gemma-3-4b-it` is a `Gemma3ForConditionalGeneration` (text decoder + SigLIP vision
   tower); the q/k/v/o and MLP names in the Qwen list also exist inside that tower, so passing the
   list would attach LoRA to vision modules a text-only task never trains.
3. **`SFTTrainer` is given `text_tokenizer`, not the bare `tokenizer`.** Unsloth returns a
   `Gemma3Processor`; handing it to TRL builds a collator exposing `.image_processor`, and
   `unsloth_zoo`'s `_is_vision_collator()` then makes `train_on_responses_only` refuse the trainer
   with *"Detected a vision data collator that does not support response-only masking"*. The trainer
   cell asserts on this rather than letting it surface a cell later.
4. **Mask markers are `<start_of_turn>user\n` / `<start_of_turn>model\n`,** not ChatML's
   `<|im_start|>...`. Those tokens do not exist in Gemma's vocabulary, so every label would be
   masked out.
5. **`.removeprefix("<bos>")` after templating.** Gemma's template emits `<bos>` itself and
   `SFTTrainer` re-tokenizes with `add_special_tokens=True`; without this every training sequence
   starts with a double BOS.
6. **`eos_token_id` must include `<end_of_turn>`.** Gemma-3 ends a turn with `<end_of_turn>`, not
   `<eos>`, and `generate()` only stops on ids listed in `eos_token_id`. Without it every sample
   burns the full token budget -- correct output, several times the runtime.
7. **No `enable_thinking` and no system slot.** Gemma's template has no thinking mode (so there is
   no `</think>` to strip, unlike the Qwen notebook) and no `system` role; it folds the system turn
   into the head of the first user turn, so the prompt text stays identical to the baseline.
8. **Gemma-3 sampling defaults:** `temperature=1.0, top_p=0.95, top_k=64` for the smoke test. Qwen's
   `0.7 / 0.8 / 20` are Qwen's own recommended values; each model uses what its authors published.

Unsloth also runs Gemma-3 in **float32** on a T4 -- float16 overflows this architecture and sm75 has
no bfloat16. That is not configurable here, and together with eager attention it is why this run
costs several times what the Qwen baseline costs per step.

## Runtime and resuming

1 epoch over 9 716 rows at effective batch 16 is ~608 optimiser steps (4 858 forward+backward
micro-batches at batch 2), plus ~30 eval passes and ~30 checkpoint writes. **Watch the progress
bar's ETA at step 20-30** -- a few minutes in it is reliable, and it is the cheapest way to find out
whether this fits Kaggle's 12h limit rather than discovering it at hour 12.

**The resume path was broken and is now fixed.** Kaggle mounts an attached notebook output under
`/kaggle/input/<slug>/`, never back into `/kaggle/working` -- which starts empty every session. The
old cell globbed only `OUTPUT_DIR` under `/kaggle/working`, found nothing every time, and silently
restarted from step 0. The train cell now searches both roots (matching on this run's directory name
so it cannot latch onto a different notebook's checkpoint), copies an input-side checkpoint into
`OUTPUT_DIR` because `/kaggle/input` is read-only, and passes it to `trainer.train()`. With
`save_steps = 20`, a timed-out session loses at most 20 steps.

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups (incl. Kaggle)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -q tabulate
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Environment and authentication

The `os.environ` values below **must** be set before `import unsloth` -- it reads them while
resolving the attention backend and the device map. If unsloth is already imported in this session,
restart the kernel first.

In [ ]:
import os

# THE OOM/SPEED FIX, and it only works before `import unsloth`. Unsloth lists gemma3 in
# _FLEX_PREFERRED_MODELS, so it selects flex_attention by default. That kernel does not compile on a
# T4 (sm75), so torch falls back to `sdpa_dense` -- the eager Python decomposition, which
# materialises the full (B, H, L, L) score matrix AND its gradient. On the with-reasoning sibling
# that produced `OutOfMemoryError: Tried to allocate 1.17 GiB` inside sdpa_dense_backward, plus
# ~0.01 it/s because the fallback runs uncompiled, op by op.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"

# Kaggle's "GPU T4 x2" exposes 2 devices. Free-tier Unsloth does not support multi-GPU training and
# HF Trainer would otherwise wrap the model in DataParallel.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# The allocator hint the OOM message itself suggests: long, highly variable sequence lengths
# fragment the pool badly, and expandable segments let freed blocks be reused at a different size.
# (PyTorch renamed this variable, so set both and let the version in use read whichever it knows.)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from huggingface_hub import login

# Optional: only needed to push the adapter to the Hub.
# On Kaggle: Add-ons > Secrets > add "HF_TOKEN", then attach it to this notebook.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(HF_TOKEN)
else:
    print("No HF_TOKEN found -- skipping login (fine unless you want to push_to_hub).")

### Load base model + LoRA adapters

In [ ]:
from unsloth import FastModel
import torch

# 2048, the same cap as qwen3-4b-stf-wo-reasoning-trace.ipynb.
#
# MEASURED on the actual de-duplicated pool with this tokenizer (10788 train / 584 valid rows;
# system + context frame + gold program, gemma-3 template, single <bos>):
#
#     train: mean 1613  p50 1585  p90 2047  p95 2243  p99 2759  max 4186
#     valid: mean 1718  p50 1728  p90 2165  p95 2311  p99 2779  max 3329
#
#     cap    train over        valid over
#     2048   1072  ( 9.9%)     94  (16.1%)   <- current
#     2560    190  ( 1.8%)     15  ( 2.6%)
#     3072     58  ( 0.5%)      2  ( 0.3%)
#     5678      0  ( 0.0%)      0  ( 0.0%)   <- previous revision
#
# Two things follow, and the second is the non-obvious one:
#
# 1. The old 5678 never truncated anything (max is 4186), so the cap itself was not what made the
#    run slow. What made it slow was evaluating the full 584-row valid set every 80 steps, no
#    length bucketing, and 486 more rows -- all fixed elsewhere in this notebook.
# 2. Because TRL pads to the longest row IN THE BATCH rather than to this cap, raising it costs
#    almost nothing in time: mean length only moves from ~1500 (at 2048, after the drops) to 1613
#    (uncapped). Going 2048 -> 2560 buys back 882 training rows and 79 valid rows for roughly 5%
#    more runtime.
#
# 2048 is kept because it is the baseline's value and this run is meant to differ from the baseline
# in the training pool alone. If you would rather keep the data than the exact alignment, 2560 is
# the better trade -- change this one number, nothing else depends on it.
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,        # QLoRA, matching the Qwen3-4B baseline
    load_in_8bit = False,
    full_finetuning = False,
    attn_implementation = "eager",   # see the env-var cell above
)

# Confirm what actually got selected. If this prints "flex_attention" the run will OOM on a T4 and
# the env var did not take effect -- restart the kernel, it is read at import time.
_cfg = model.config
_impl = getattr(_cfg, "_attn_implementation", None) or getattr(_cfg, "attn_implementation", None)
print(f"attn implementation = {_impl}")
print(f"dtype = {next(model.parameters()).dtype}")
assert _impl != "flex_attention", (
    "flex_attention is selected and will OOM on a T4. Restart the kernel so "
    "UNSLOTH_ENABLE_FLEX_ATTENTION=0 is read before unsloth is imported."
)

In [ ]:
# LoRA targets via the finetune_* flags instead of the Qwen notebook's explicit target_modules list.
# Those exact module names (q/k/v/o_proj, gate/up/down_proj) also exist inside the SigLIP vision
# tower, so passing the list would attach adapters to vision modules a text-only task never trains,
# burning VRAM and leaving dead weights in the checkpoint. finetune_vision_layers=False excludes them
# properly, and the three language flags expand to exactly the seven projections Qwen listed.
#
# r / lora_alpha / dropout / bias / random_state are identical to the Qwen3-4B baseline.
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # text-only task -- skip the SigLIP tower
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 32,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# Fail loudly if LoRA attached to nothing (wrong module names => a silent no-op training run).
_n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert _n_trainable > 0, "No trainable parameters -- LoRA did not attach to any module."
print(f"Trainable parameters: {_n_trainable:,}")

In [ ]:
from unsloth.chat_templates import get_chat_template

# Pin the Gemma-3 chat template explicitly, as Unsloth's own Gemma3 notebook does. This guarantees
# apply_chat_template emits
#   <start_of_turn>user ... <end_of_turn>\n<start_of_turn>model ...
# which is exactly what train_on_responses_only masks on below, and it is saved next to the adapter
# so the eval notebook builds byte-identical prompts.
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

# For gemma-3-4b-it, `tokenizer` is a Gemma3Processor (image+text), not a bare tokenizer. Its
# __call__ signature starts with `images`, so tokenizer(text) silently binds the string to the wrong
# argument. Keep a handle on the real text tokenizer for the trainer, counting and decoding.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
print(f"tokenizer: {type(tokenizer).__name__}  ->  text_tokenizer: {type(text_tokenizer).__name__}")

# Gemma-3 ends a turn with <end_of_turn>, not <eos>, and generate() only stops on ids listed in
# eos_token_id. Only the smoke test needs this here; the eval notebook needs it for all 497 samples.
_EOS_IDS = []
for _t in (text_tokenizer.eos_token_id, text_tokenizer.convert_tokens_to_ids("<end_of_turn>")):
    if _t is not None and _t not in _EOS_IDS:
        _EOS_IDS.append(_t)
print(f"_EOS_IDS = {_EOS_IDS}")

<a name="Data"></a>
### Data prep

Train is the concatenation of **`ViNumQA/train.json` (2 993, Vietnamese)** and
**`FinQA_processed.json` (8 281, English)**, then **minus the 486 rows whose `id` also appears in
valid or test** = 10 788 rows. Valid (584) and test (497) stay ViNumQA-only. The two files carry the
same record schema, so a single `process_split()` handles both and the context formatting and system
prompt used by the 0-shot / 1-shot / SFT notebooks are unchanged -- the formatting and prompt cells
are still copied verbatim from `qwen3-4b-stf-wo-reasoning-trace.ipynb`.

The de-duplication is what makes this row's test PA/EA comparable to the other Table 2 rows. Earlier
revisions printed the overlap and trained on it anyway; the cell now drops it and asserts the
overlap is empty afterwards.

`qa.reasoning_trace` is never read: the assistant label is the bare gold program.

In [ ]:
import glob
import pandas as pd
from pathlib import Path
from tabulate import tabulate

# --- ViNumQA: supplies train (part 1), valid and test -------------------------------------------
# The explicit path is the one the Qwen notebook hardcodes; the glob fallback finds the files
# wherever the dataset happens to be mounted.
_VINUM_CANDIDATES = [
    Path("/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa"),
    Path("/kaggle/input/vlsp2025-vinumqa"),
    Path("datasets/ViNumQA"),   # local repo path, if running outside Kaggle
]
DATA_DIR = next((p for p in _VINUM_CANDIDATES if (p / "train.json").exists()), None)
if DATA_DIR is None:
    _hits = glob.glob("/kaggle/input/**/train.json", recursive=True)
    DATA_DIR = Path(_hits[0]).parent if _hits else None
if DATA_DIR is None:
    raise FileNotFoundError(
        "train.json not found -- attach the ViNumQA dataset (ntphuc149x2/vlsp2025-vinumqa) "
        "as a data source, or add its path to _VINUM_CANDIDATES above."
    )
print("ViNumQA data from:", DATA_DIR)

# --- FinQA: supplies train (part 2) --------------------------------------------------------------
# Same record schema as ViNumQA (id / pre_text / table / post_text / qa{question, program, exe_ans}),
# so it flows through process_split() below unchanged. It is the untranslated English source corpus;
# ViNumQA is its Vietnamese rendering. Kaggle may mount it beside ViNumQA or as its own data source,
# hence the candidate list plus a recursive glob.
_FINQA_CANDIDATES = [
    DATA_DIR / "FinQA_processed.json",
    Path("/kaggle/input/finqa-processed/FinQA_processed.json"),
    Path("/kaggle/input/finqa/FinQA_processed.json"),
    Path("datasets/FinQA_processed.json"),   # local repo path, if running outside Kaggle
]
FINQA_PATH = next((p for p in _FINQA_CANDIDATES if p.exists()), None)
if FINQA_PATH is None:
    _hits = glob.glob("/kaggle/input/**/FinQA_processed.json", recursive=True)
    FINQA_PATH = Path(_hits[0]) if _hits else None
if FINQA_PATH is None:
    raise FileNotFoundError(
        "FinQA_processed.json not found -- attach it as a data source, or add its path to "
        "_FINQA_CANDIDATES above."
    )
print("FinQA data from:  ", FINQA_PATH)

vinum_train_df = pd.read_json(DATA_DIR / "train.json")
finqa_train_df = pd.read_json(FINQA_PATH)
valid_df = pd.read_json(DATA_DIR / "valid.json")
test_df = pd.read_json(DATA_DIR / "test.json")

# Fail early if the two corpora ever diverge in schema -- a missing column would otherwise surface
# hundreds of rows later as a KeyError inside process_split().
_REQUIRED_COLS = {"id", "pre_text", "table", "post_text", "qa"}
for _name, _df in [("ViNumQA train", vinum_train_df), ("FinQA", finqa_train_df)]:
    _missing = _REQUIRED_COLS - set(_df.columns)
    assert not _missing, f"{_name} is missing columns {_missing} -- schemas must match to concatenate."

# Train = ViNumQA train + FinQA, as one pool. `source` is carried only for the report below; it is
# dropped by process_split(), so nothing downstream can accidentally condition on it.
vinum_train_df["source"] = "vinumqa"
finqa_train_df["source"] = "finqa"
train_df = pd.concat([vinum_train_df, finqa_train_df], ignore_index=True)

print(f"\npooled train={len(train_df)} (ViNumQA {len(vinum_train_df)} + FinQA {len(finqa_train_df)}), "
      f"valid={len(valid_df)}, test={len(test_df)}")

assert (len(train_df), len(valid_df), len(test_df)) == (11274, 584, 497), (
    f"Unexpected split sizes {(len(train_df), len(valid_df), len(test_df))} -- expected "
    "(11274, 584, 497) = ViNumQA train 2993 + FinQA 8281, valid 584, test 497. Check that both "
    "source files are the full ones, not a *_with_reasoning_trace_* subset."
)

# ---------------------------------------------------------------------------------------------
# DE-DUPLICATION -- the contamination is now REMOVED, where earlier revisions only printed it.
#
# ViNumQA is a Vietnamese translation of FinQA and the two share `id`s, so a large slice of the
# eval splits had an English twin inside FinQA: same table, same question, same gold program (210
# of 497 test rows, 276 of 584 valid rows). Training on them made test PA/EA partly a measurement
# of memorised training items, not comparable to the other Table 2 rows, and it kept eval_loss
# falling for reasons unrelated to progress.
#
# Both problems go away by dropping the overlap. `source` is kept until after the report so the
# breakdown shows which corpus lost the rows.
# ---------------------------------------------------------------------------------------------
_leak_ids = set(valid_df["id"]) | set(test_df["id"])
_before = len(train_df)
_dropped = train_df[train_df["id"].isin(_leak_ids)]
train_df = train_df[~train_df["id"].isin(_leak_ids)].reset_index(drop=True)

print(
    f"\nDE-DUPLICATION vs valid+test:\n"
    f"  dropped {_before - len(train_df)} of {_before} pooled train rows "
    f"({dict(_dropped['source'].value_counts())})\n"
    f"  train is now {len(train_df)} rows "
    f"(ViNumQA {(train_df['source'] == 'vinumqa').sum()} + FinQA {(train_df['source'] == 'finqa').sum()})"
)

assert len(train_df) == 10788, (
    f"Expected 10788 rows after de-duplication, got {len(train_df)}. The overlap between FinQA and "
    "ViNumQA valid/test changed -- re-check the input files before reporting numbers."
)
assert not (set(train_df["id"]) & _leak_ids), "De-duplication did not remove every overlapping id."
print("  no train id appears in valid or test.")

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["table_raw"] = df["table"]  # keep the raw rows for table_* row-name lookup at eval time
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question",
             "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]
    return df

train_df = process_split(train_df)
valid_df = process_split(valid_df)
test_df = process_split(test_df)
test_df["generated_program"] = ""

train_df.sample(n=3)

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

# Same prompt format as the 0-shot/1-shot/sft notebooks, so results are
# comparable across all experiments (only the training regime differs).

### Build the conversational dataset

In [ ]:
def build_conversation(row):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    # The assistant turn is the bare gold program: no <think> block, no reasoning trace. The system
    # turn is kept even though Gemma has no system slot -- its template folds the content into the
    # head of the first user turn, so the prompt text stays identical to the Qwen baseline, and the
    # template renders the "assistant" role as <start_of_turn>model itself.
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": str(row["program"]).strip()},
    ]


def make_text_dataset(df):
    conversations = [build_conversation(row) for _, row in df.iterrows()]
    # No enable_thinking flag: Gemma's chat template has no such concept (the Qwen notebook passes
    # enable_thinking=False to suppress an empty <think></think> pair; there is nothing to suppress
    # here).
    #
    # 1. .removeprefix("<bos>") -- REQUIRED, and done the same way in Unsloth's own Gemma3 notebook.
    #    Gemma's template emits <bos> itself, and SFTTrainer then tokenizes this text with
    #    add_special_tokens=True, prepending a SECOND <bos>.
    # 2. One conversation per call instead of the whole list: the processor's batched
    #    apply_chat_template path behaves differently across transformers versions.
    return [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False,
        ).removeprefix("<bos>")
        for convo in conversations
    ]


train_texts = make_text_dataset(train_df)
valid_texts = make_text_dataset(valid_df)
print(len(train_texts), len(valid_texts))

assert not train_texts[0].startswith("<bos>"), "BOS was not stripped -- training text would double it."
assert "<start_of_turn>model" in train_texts[0], "Unexpected template output for Gemma-3."

print("\n" + "=" * 26 + " templated sample " + "=" * 26)
print(train_texts[0][:1200])

In [ ]:
# How much of the data actually fits in MAX_SEQ_LENGTH? At the baseline's 2048 this is no longer a
# footnote: the rows above the cap are DROPPED by the next cell rather than truncated, so this is the
# cell that decides how much training data the run actually sees. Report the numbers it prints.
#
# Why dropped and not truncated: TRL truncates from the right, and the assistant turn -- the gold
# program, the only span train_on_responses_only computes loss on -- is at the very end. A truncated
# row arrives with every label masked to -100, and a batch (size 2) where both rows are like that
# produces a NaN mean cross-entropy that the LoRA weights never recover from.
#
# This cell tokenizes all ~10.8k training texts one at a time, so it takes a couple of minutes.
#
# text_tokenizer, not tokenizer: the latter is a Gemma3Processor whose __call__ takes `images` first.
import numpy as np


def compute_token_lengths(texts):
    return np.array([len(text_tokenizer(t, add_special_tokens=True)["input_ids"]) for t in texts])


train_lengths = compute_token_lengths(train_texts)
valid_lengths = compute_token_lengths(valid_texts)

for name, lengths in [("train", train_lengths), ("valid", valid_lengths)]:
    over = int((lengths > MAX_SEQ_LENGTH).sum())
    print(f"{name}: n={len(lengths)}  max={lengths.max()}  p99={np.percentile(lengths, 99):.0f}  "
          f"p95={np.percentile(lengths, 95):.0f}  mean={lengths.mean():.0f}  "
          f"over MAX_SEQ_LENGTH({MAX_SEQ_LENGTH})={over} ({100 * over / len(lengths):.1f}%)")

_overall_max = int(max(train_lengths.max(), valid_lengths.max()))
print(f"\nOverall max token length: {_overall_max}")
print("Rows above the cap are removed by the next cell -- the 'over' counts above are what gets "
      "dropped, and belong in the write-up next to PA/EA.")

In [ ]:
from datasets import Dataset

# --- Drop rows that do not fit MAX_SEQ_LENGTH ----------------------------------------------------
# See the length-stats cell for why this is a drop and not a truncation: the gold program sits at the
# end of the sequence, TRL truncates from the right, and a row whose program is cut off arrives with
# every label masked to -100. Two such rows in one batch of 2 make the mean cross-entropy NaN.
#
# Valid rows are filtered too -- they are only forward-passed for eval_loss, but an all-masked batch
# gives a NaN there as well, and with eval_loss now driving both checkpoint selection and early
# stopping, a NaN there decides which model you keep.
def _drop_over_length(texts, lengths, name):
    keep = lengths <= MAX_SEQ_LENGTH
    n_dropped = int((~keep).sum())
    kept = [t for t, ok in zip(texts, keep) if ok]
    print(f"{name}: dropped {n_dropped} of {len(texts)} rows over MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} "
          f"-> {len(kept)} kept ({100 * n_dropped / len(texts):.1f}% lost)")
    assert kept, f"Every {name} row is longer than MAX_SEQ_LENGTH -- the cap is set far too low."
    return kept


train_texts_fit = _drop_over_length(train_texts, train_lengths, "train")
valid_texts_fit = _drop_over_length(valid_texts, valid_lengths, "valid")

# --- THE CHEAPEST RUNTIME LEVER IN THIS NOTEBOOK -------------------------------------------------
# Cap the eval set, or None to use all of it.
#
# This knob matters again now that the trainer evaluates every 20 steps. Over ~600 steps that is ~30
# full passes over the valid set -- roughly +40% on top of training time, the single largest cost
# after training itself. Subsampling is the cheapest way to buy that back:
#
#     None (490 rows) -> ~245 forward batches per eval  x30 = ~7350
#     256             -> ~128                           x30 = ~3840   (-48%)
#     128             ->  ~64                           x30 = ~1920   (-74%)
#
# It costs almost nothing statistically: the subset is drawn once with a fixed seed, so every eval
# scores the SAME rows and eval_loss stays a consistent early-stopping and checkpoint-selection
# signal -- it just has slightly more variance as an absolute number. Report which value you used.
#
# Left at None so the run matches the config as specified. Set 128 here if the projection printed by
# the trainer cell says the run will not fit.
EVAL_SUBSET_SIZE = None

# .shuffle() matters: train_df is a concat, so unshuffled it would feed the Vietnamese ViNumQA rows
# first and only then the English FinQA ones -- the LR schedule would decay across a hard language
# boundary rather than across a mixed stream. (group_by_length in the trainer reorders batches by
# length afterwards, but only within the shuffled stream.)
train_dataset = Dataset.from_dict({"text": train_texts_fit}).shuffle(seed=3407)
valid_dataset = Dataset.from_dict({"text": valid_texts_fit})
if EVAL_SUBSET_SIZE is not None and EVAL_SUBSET_SIZE < len(valid_dataset):
    valid_dataset = valid_dataset.shuffle(seed=3407).select(range(EVAL_SUBSET_SIZE))
    print(f"Eval set subsampled to {len(valid_dataset)} of {len(valid_texts_fit)} samples.")

train_dataset, valid_dataset

<a name="Train"></a>
### Train

The loss is masked to the assistant turn only (`train_on_responses_only`), so the model is not
penalised for "predicting" the system prompt, the context or the question -- important here because
the context block dwarfs the program string we want it to learn.

For Gemma the markers are `<start_of_turn>user\n` / `<start_of_turn>model\n`, **not** ChatML's
`<|im_start|>`. The masking cell asserts a non-empty span survived and prints it, so a template
mismatch fails in seconds instead of silently training on nothing for hours.

It then scans 500 random training examples for the opposite failure: a row where *every* label is
masked, which is what a sequence truncated before `<start_of_turn>model` looks like. Two of those in
one batch of 2 give a `NaN` mean cross-entropy and the LoRA weights never recover -- silently, hours
in. The dataset cell already drops over-length rows, so this scan should find none; it exists because
`SFTTrainer` tokenizes independently of that cell.

In [ ]:
import math

from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

OUTPUT_DIR = "/kaggle/working/gemma3-4b-vinumqa-sft-wo-reasoning"

# Pure throughput, no effect on which samples are seen or on the schedule. batch=2 pads to the longer
# of the pair, and after .shuffle() that partner is random, so a short program row routinely gets
# padded out to a long one. group_by_length buckets similar lengths together instead; it changes
# which rows share a batch (so gradient noise differs slightly from the Qwen baseline) but nothing
# else. Set False to drop it -- costs roughly 15-20% more runtime.
GROUP_BY_LENGTH = True

trainer = SFTTrainer(
    model=model,
    # text_tokenizer, NOT the bare `tokenizer`. This is the one line kept from the previous revision
    # rather than taken verbatim from the pasted config, because it is not a hyperparameter: for
    # gemma-3-4b-it Unsloth returns a Gemma3Processor, TRL then builds a collator holding it, and
    # unsloth_zoo's _is_vision_collator() flags anything exposing .image_processor -- so
    # train_on_responses_only in the NEXT cell hard-fails with "Detected a vision data collator that
    # does not support response-only masking". The assert below catches it either way.
    tokenizer=text_tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,

        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,

        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_ratio=0.03,

        # Evaluate every 20 steps
        eval_strategy="steps",
        eval_steps=20,

        # Save checkpoint every 20 steps
        save_strategy="steps",
        save_steps=20,

        logging_steps=20,

        # Load best checkpoint
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        save_total_limit=2,

        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="none",

        group_by_length=GROUP_BY_LENGTH,
        dataset_num_proc=2,
    ),

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,      # Dừng sau 3 lần eval không cải thiện
            early_stopping_threshold=0.0    # Chỉ cần loss giảm là được
        )
    ],
)

# Fail here, not in the next cell, if the collator still carries the processor.
_coll = trainer.data_collator
_coll_tok = getattr(_coll, "processor", None) or getattr(_coll, "tokenizer", None)
print(f"data_collator = {type(_coll).__name__}, its tokenizer = {type(_coll_tok).__name__}")
assert not hasattr(_coll_tok, "image_processor"), (
    "The collator is holding a processor -- train_on_responses_only will reject it. "
    "Check that tokenizer = text_tokenizer above."
)

# --- What this schedule actually costs, printed before committing hours to it -------------------
# eval_steps=20 over ~600 steps means ~30 full passes over the valid set. That is the dominant cost
# in this cell: each pass is len(valid)/2 forward batches, and a forward is roughly a quarter of a
# training micro-batch (which pays forward + gradient-checkpoint recompute + backward). At the
# measured lengths that works out to about +40% on top of training time.
#
# Two levers if the printed projection is too high, in order of how little they cost the experiment:
#   1. EVAL_SUBSET_SIZE in the dataset cell -- 128 instead of None cuts eval cost by ~74% and still
#      gives a usable early-stopping signal, because the subset is identical at every eval.
#   2. eval_steps / save_steps 20 -> 150: 4 evals instead of 30, and patience=3 then means 450 steps
#      without improvement rather than 60.
_steps_per_epoch = math.ceil(len(train_dataset) / (2 * 8))
_n_evals = _steps_per_epoch // 20
_eval_batches = math.ceil(len(valid_dataset) / 2)
print(f"\n~{_steps_per_epoch} optimiser steps for 1 epoch over {len(train_dataset)} samples "
      f"({_steps_per_epoch * 8} forward+backward micro-batches at batch 2).")
print(f"eval every 20 steps => ~{_n_evals} evals x {_eval_batches} batches "
      f"= ~{_n_evals * _eval_batches} extra forward passes on {len(valid_dataset)} valid samples.")
print(f"~{_steps_per_epoch // 20} checkpoint writes (save_total_limit=2 keeps the best + newest).")

# EarlyStoppingCallback(patience=3) at eval_steps=20 stops after 60 steps without improvement --
# about 10% of this epoch. Precedent from the sibling notebook: on the `pa_en_finqa` variant the same
# settings stopped at step 220 of 456 with the best checkpoint at step 160, i.e. the run trained on
# ~35% of the data. If that happens here, the log line below is where to look for it.
print(f"\nEarlyStopping: patience=3 x eval_steps=20 => stops after 60 steps "
      f"({100 * 60 / _steps_per_epoch:.0f}% of the epoch) without an eval_loss improvement.")

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# THE Gemma-specific fix. In the Qwen notebook these two strings are "<|im_start|>user\n" and
# "<|im_start|>assistant\n" -- ChatML markers. Gemma's tokenizer has no such tokens, so nothing in
# the templated text would ever match and every label would be masked out (loss 0/NaN, or the masker
# errors outright). Gemma-3 turns are delimited by <start_of_turn>, and the assistant role is
# spelled "model".
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

# Sanity check before burning hours of GPU time: confirm the mask kept some labels, and that what
# survived is the assistant turn (the program) rather than the system prompt or the context.
_ex = trainer.train_dataset[0]
_kept = [t for t, l in zip(_ex["input_ids"], _ex["labels"]) if l != -100]
assert _kept, "train_on_responses_only masked EVERYTHING -- instruction/response parts do not match the template."
_kept_text = text_tokenizer.decode(_kept)
print(f"unmasked label tokens: {len(_kept)} / {len(_ex['input_ids'])}")
print("--- trained-on span (this is what the loss is computed on) ---")
print(repr(_kept_text))
assert "### CONTEXT" not in _kept_text and "LIST OF 10 VALID OPERATORS" not in _kept_text, (
    "The prompt is being trained on -- the mask boundary is wrong."
)

# Second check, and the one that actually protects this run: no example may end up with ZERO
# unmasked labels. That happens when truncation at MAX_SEQ_LENGTH cuts the sequence before
# <start_of_turn>model, and a batch (size 2) of two such rows makes the mean cross-entropy NaN --
# which destroys the LoRA weights silently, hours in. The dataset cell drops over-length rows so
# this should find nothing; it is here because SFTTrainer tokenizes independently of that cell and a
# row sitting exactly on the boundary is the kind of thing that slips through.
import random

_rng = random.Random(3407)
_sample_idx = _rng.sample(range(len(trainer.train_dataset)), min(500, len(trainer.train_dataset)))
_empty = [i for i in _sample_idx if not any(l != -100 for l in trainer.train_dataset[i]["labels"])]
print(f"\nchecked {len(_sample_idx)} random training examples: {len(_empty)} with no unmasked label")
assert not _empty, (
    f"{len(_empty)} of {len(_sample_idx)} sampled rows have every label masked -- their assistant "
    f"turn was truncated away. Lower the row-length filter or raise MAX_SEQ_LENGTH; training as-is "
    f"will produce NaN losses. Offending dataset indices: {_empty[:10]}"
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Train. **The resume path below is the fixed one.** The previous revision globbed `checkpoint-*` under
`OUTPUT_DIR` only -- but `OUTPUT_DIR` lives in `/kaggle/working`, which starts empty in every session,
and an attached notebook output is mounted read-only under `/kaggle/input/<slug>/`. So the glob found
nothing every time and the run silently restarted from step 0, which is how a 12h timeout cost the
whole run rather than a few hundred steps.

The cell now searches `/kaggle/working` **and** `/kaggle/input`, keeps only checkpoints sitting in a
directory named after this run (so it cannot latch onto a different notebook's adapter that happens
to be attached), copies the newest one into `OUTPUT_DIR` because `/kaggle/input` is read-only, and
hands it to `trainer.train()`. With `save_steps = 20` a timed-out session loses at most 20 steps.

To continue a timed-out session: *Add Input > Your Work > Notebooks*, attach this notebook's own
previous version, then run top to bottom.

In [ ]:
import re
import shutil
from pathlib import Path

# Only checkpoints written by THIS run count. Both roots are searched for
# ".../<RUN_NAME>/checkpoint-<step>", so an unrelated adapter that happens to be attached as a data
# source (the w-reasoning one, say) cannot be picked up and resumed from.
_RUN_NAME = Path(OUTPUT_DIR).name


def _find_checkpoints():
    """(step, path) for every resumable checkpoint, oldest first."""
    found = []
    _roots = [
        (Path(OUTPUT_DIR), "checkpoint-*"),          # this session, if the cell is re-run
        (Path("/kaggle/input"), f"**/{_RUN_NAME}/checkpoint-*"),  # a previous session, attached
    ]
    for root, pattern in _roots:
        if not root.exists():
            continue
        for p in root.glob(pattern):
            # trainer_state.json carries the step count, LR schedule and RNG state. Without it
            # resume_from_checkpoint restarts the schedule, so a directory lacking it is not
            # resumable and is skipped rather than half-used.
            if p.is_dir() and (p / "trainer_state.json").exists():
                m = re.search(r"checkpoint-(\d+)$", p.name)
                if m:
                    found.append((int(m.group(1)), p))
    return sorted(found, key=lambda t: t[0])


_ckpts = _find_checkpoints()
for _step, _p in _ckpts:
    print(f"  found step {_step:>5}  {_p}")

RESUME_CHECKPOINT = None
if _ckpts:
    _step, _src = _ckpts[-1]
    # /kaggle/input is read-only and the Trainer writes its next checkpoint into output_dir, so the
    # resumed one has to be local first.
    if Path("/kaggle/input") in _src.parents:
        _dst = Path(OUTPUT_DIR) / _src.name
        if not _dst.exists():
            print(f"Copying {_src} -> {_dst} (/kaggle/input is read-only)")
            _dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(_src, _dst)
        _src = _dst
    RESUME_CHECKPOINT = str(_src)

print("\nResume checkpoint:", RESUME_CHECKPOINT or "none -- starting a new run")

trainer_stats = trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)

<a name="Save"></a>
### Save the adapter -- before anything optional runs

The adapter is the deliverable. On Kaggle a Save Version run that raises *anywhere* is marked failed,
and a failed run does not publish `/kaggle/working` at all -- so a late crash in an optional cell
would discard an adapter that is already on disk. Save first; wrap everything after it.

`load_best_model_at_end = True` means the in-memory model already *is* the checkpoint with the lowest
`eval_loss`, so saving from here is correct -- not a hand-picked `checkpoint-N`.

**Then check which step that was.** The run evaluates every 20 steps with
`EarlyStoppingCallback(patience=3)`, so it stops after 60 steps without an improvement -- about 10 %
of the epoch. The **stats cell below the save** prints `global_step` against the planned step count
and the best checkpoint's path, which is what says whether the adapter came from a full epoch or from
an early stop a third of the way in. The sibling notebook's `pa_en_finqa` run stopped at step 220 of
456 with the best at 160.

In [ ]:
import traceback

ADAPTER_DIR = "/kaggle/working/gemma3-4b-vinumqa-sft-wo-reasoning-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)   # the Gemma3Processor: tokenizer + chat template + config

# Prove the adapter is really on disk. save_pretrained on a PEFT model writes adapter_config.json +
# adapter_model.safetensors; a config-only directory means the save silently did nothing, and the
# eval notebook has nothing to attach.
_files = sorted(os.listdir(ADAPTER_DIR))
print(f"Saved LoRA adapter to {ADAPTER_DIR}")
for _f in _files:
    print(f"  {_f:44s} {os.path.getsize(os.path.join(ADAPTER_DIR, _f)) / 1e6:8.2f} MB")
assert "adapter_config.json" in _files and any(f.startswith("adapter_model.") for f in _files), (
    f"No adapter weights in {ADAPTER_DIR} -- got {_files}. Nothing usable was saved."
)
assert any(f.startswith("chat_template") for f in _files) or "tokenizer_config.json" in _files, (
    "No chat template saved next to the adapter -- the eval notebook would build prompts without "
    "the <start_of_turn> markers this run trained on."
)

# Optional Hub push. /kaggle/working is only published when the whole Save Version run finishes, so a
# run that hits the 12h limit loses its output entirely; the Hub copy survives that. Wrapped because
# it is the only step here that talks to a remote.
HF_REPO_ID = os.environ.get("HF_REPO_ID", "").strip()
if HF_TOKEN:
    try:
        if not HF_REPO_ID:
            from huggingface_hub import whoami
            HF_REPO_ID = f"{whoami(token=HF_TOKEN)['name']}/gemma3-4b-vinumqa-sft-wo-reasoning-adapter"
        model.push_to_hub(HF_REPO_ID, token=HF_TOKEN, private=True)
        tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN, private=True)
        print(f"Pushed adapter to https://huggingface.co/{HF_REPO_ID}")
    except Exception:
        print("HUB PUSH FAILED -- the local save above is intact, continuing.\n"
              "Most likely a read-only HF_TOKEN: regenerate it with WRITE access.\n")
        print(traceback.format_exc())
else:
    print("Skipping Hub push (needs the HF_TOKEN secret).")

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB ({round(used_memory / max_memory * 100, 3)} % of max).")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")

# --- Did EarlyStoppingCallback cut the run short? ------------------------------------------------
# patience=3 at eval_steps=20 fires after 60 steps without an eval_loss improvement, so this run can
# end long before the epoch does -- and the saved adapter would then have seen only part of the data.
# The numbers below are what says so, and they belong in the write-up next to PA/EA.
_ran = trainer.state.global_step
_planned = math.ceil(len(train_dataset) / (2 * 8))
_best = trainer.state.best_model_checkpoint
print(f"\nSteps run: {_ran} / {_planned} planned ({100 * _ran / _planned:.0f}% of the epoch)")
print(f"Best checkpoint by eval_loss: {_best}")
print(f"Best eval_loss: {trainer.state.best_metric}")
if _ran < _planned:
    print(f"\nEARLY STOPPED at step {_ran} of {_planned}. The adapter saved below is the best "
          f"checkpoint by eval_loss, but it was trained on roughly "
          f"{100 * _ran / _planned:.0f}% of the {len(train_dataset)}-row pool -- state this when "
          f"reporting, and see the trainer cell for the two levers (EVAL_SUBSET_SIZE, or "
          f"eval_steps 20 -> 150 so patience=3 means 450 steps instead of 60).")
else:
    print("\nRan the full epoch -- early stopping did not fire.")

<a name="Inference"></a>
### Smoke test (optional)

One streamed sample, to eyeball that the adapter emits a bare program -- no `<think>`, no prose --
and stops at `<end_of_turn>`. Diagnostic only: it runs after the save and swallows its own
exceptions, so an OOM here cannot cost you the adapter or fail the Save Version run.

In [ ]:
try:
    from transformers import TextStreamer

    FastModel.for_inference(model)

    _row = test_df.iloc[0]
    _prompt = USER_MESSAGE_FRAME.format(
        pre_text=_row["pre_text"], table=_row["table"],
        post_text=_row["post_text"], question=_row["question"],
    )
    _messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": _prompt},
    ]

    # Template to TEXT first, then tokenize -- two separate steps, deliberately.
    # tokenizer.apply_chat_template(..., tokenize=True) does NOT work here: on a Gemma3Processor that
    # path is multimodal-only, iterating message["content"] expecting a list of typed parts, and it
    # raises "TypeError: string indices must be integers" on plain strings. The dataset builder above
    # gets away with the same call only because that whole block sits behind `if tokenize:`.
    #
    # add_special_tokens=False because the template already emits <bos> -- this reproduces exactly
    # the single-BOS sequence SFTTrainer trained on.
    _text = tokenizer.apply_chat_template(_messages, tokenize=False, add_generation_prompt=True)
    _inputs = text_tokenizer(_text, return_tensors="pt", add_special_tokens=False).to(model.device)

    _ = model.generate(
        **_inputs,
        max_new_tokens = 256,                          # same cap as the Qwen baseline
        do_sample = True,
        temperature = 1.0, top_p = 0.95, top_k = 64,   # Gemma-3's recommended settings
        eos_token_id = _EOS_IDS,                       # stop at <end_of_turn>, not only <eos>
        streamer = TextStreamer(text_tokenizer, skip_prompt = True),
    )
    print("\nGold program:", _row["program"], "| Gold answer:", _row["answer"])

except Exception:
    print("SMOKE TEST FAILED -- ignored on purpose. The adapter is already saved above.\n")
    print(traceback.format_exc())

## Done -- next step

`/kaggle/working/gemma3-4b-vinumqa-sft-wo-reasoning-adapter` is the artifact. Commit with **Save Version**, then in
`gemma3-4b-stf-wo-reasoning-eval-only.ipynb` add this notebook as a data source
(*Add Input > Your Work > Notebooks*) to score PA/EA.

Note that `/kaggle/working/gemma3-4b-vinumqa-sft-wo-reasoning` (no `-adapter`) is the trainer's `output_dir`:
intermediate `checkpoint-*` folders capped at 2 by `save_total_limit`. It is not what the eval
notebook wants -- the `-adapter` directory is -- but **keep it**: it is what the resume cell reads if
this run has to be continued in another session.

### What to state when reporting this row of Table 2

The contamination caveat is gone -- the 486 overlapping rows are dropped at the data cell and the
notebook asserts it. What still needs saying:

1. **Training pool**: ViNumQA train + FinQA, 10 788 rows after de-duplication and 9 716 after
   dropping rows longer than `MAX_SEQ_LENGTH`, against the Qwen baseline's ViNumQA-only 2 993. This
   is the variable under test.
2. **1 epoch, not the baseline's 3** -- and **how much of that epoch actually ran.** The schedule
   evaluates every 20 steps with `EarlyStoppingCallback(patience=3)`, which fires after 60 steps
   without improvement, so the run can end well short of the epoch. The stats cell prints
   `global_step / planned`; quote it. A number produced by a run that stopped at 35 % of the data is
   not comparable to one that consumed the whole file, and the Qwen baseline has no early stopping
   at all, so it always consumes its whole file.
3. **`MAX_SEQ_LENGTH = 2048`** matches the baseline, and the two drop counts the run prints (486
   contamination, ~1 072 over-length) define what "10 788" actually became.

Everything else in the schedule -- effective batch 16, lr 2e-4, cosine, warmup 0.03, `adamw_8bit`,
weight decay 0.001, seed 3407, LoRA r32/alpha32 -- matches the baseline.